# 4 · Variants, and the result that did not work

This notebook is the awkward one, and it is meant to be.

The project was built to predict whether a PIEZO1 variant causes **gain** or
**loss** of function from structure. That is the whole point of the machinery
in the other three notebooks. It does not work — five pre-registered tests,
five different predictor families, five nulls — and, more usefully, the data
that would settle it does not exist and cannot be assembled.

If you take the machinery and skip this, you will repeat a mistake this project
already made five times.

In [ ]:
from piezo1.core.annotations import load_annotations

ann = load_annotations("human")
by_class = {}
for v in ann.variants:
    by_class.setdefault(v.classification, []).append(v)

print(f"{len(ann.variants)} curated variants, every wild-type residue verified")
for name, group in sorted(by_class.items(), key=lambda kv: -len(kv[1])):
    print(f"  {name:24s} {len(group):3d}")

## Coverage, reported rather than hidden

Most variants of interest are not resolved in any deposited human structure.
All six human entries model from residue 570 only. A viewer that highlighted
nothing would be indistinguishable from one that had found nothing, so the
count is stated.

In [ ]:
unresolved = [v for v in ann.variants if not v.modelled_in]
print(f"{len(unresolved)} of {len(ann.variants)} variants are resolved in NO "
      f"human structure")
print("among them:", ", ".join(sorted(v.label for v in unresolved)[:6]), "...")

## What a prediction is worth

`prediction_record` holds the record as data rather than prose, so the GUI, the
command line, the tour and this notebook cannot drift apart. Each entry is a
test that was **pre-registered** — hypothesis, statistic and decision rule
fixed and committed *before* the comparison was run.

In [ ]:
from piezo1.analysis.prediction_record import ALL_PREREGISTERED, headline

print(headline())
print()
for e in ALL_PREREGISTERED:
    p = "  n/a " if e.p_value is None else f"{e.p_value:6.3f}"
    print(f"  Round {e.round:2d}  delta {e.cliffs_delta:+.3f}  p {p}  "
          f"n {e.n_gof:2d}/{e.n_lof:2d}  {e.predictor}")

## Why five nulls are not "nearly something"

One of them, Round 41, returned p = 0.0477 — below the conventional 0.05. It is
still a null, and it was declared one, because the decision rule fixed in
advance had **three** clauses: p below threshold, effect in the predicted
direction, *and* a confidence interval excluding zero. The interval spanned
zero, so the rule was not met.

Deciding that after seeing the number is how a null becomes a finding.

In [ ]:
from piezo1.analysis.prediction_record import what_it_means

for line in what_it_means():
    print("*", line)

## The part that makes this more than a list of failures

A null result normally means *get more data*. Here the amount of data that
would be needed was costed, and it exceeds what could ever exist.

**Across positions:** the effect the best predictor produces would need 134
directional variants. The ceiling — every curated variant, every ClinVar entry
with an inferable direction, plus everything the literature harvest could add —
is 59.

In [ ]:
from piezo1.analysis.feasibility import assess

report = assess(n_simulations=300)
print(report.summary())

**Within positions:** comparing two variants at the *same* residue removes the
between-position variance, which consumed 99.8% of the first predictor's
signal. It is the obvious way out, and it is closed too: the design needs 8
shared positions even at an implausibly good predictor, and the curated and
ClinVar sets together contain exactly **one**.

In [ ]:
from piezo1.analysis.data_routes import evidence_summary

for key, value in evidence_summary().items():
    print(f"  {key:34s} {value}")

## So what is reusable?

Not the predictor. The apparatus that established it could not be validated:

* **pre-registration** with a decision rule fixed in advance, so a marginal p
  cannot be promoted after the fact;
* a **negative control** in every test — in Round 48 the control out-performed
  every mechanistic endpoint, which is what a null looks like from the inside;
* **feasibility costed before another attempt**, which is what turned "we need
  more data" into "the data that could exist is not enough";
* every **checking instrument calibrated** against a known answer before its
  disagreement is believed. Six times in this project, the instrument built to
  check the pipeline was itself the thing at fault.

`docs/METHODS_NOTE.md` writes this up for someone else's project.

In [ ]:
# The coupling score still exists and still computes. What was removed in
# Round 58 is the claim that its sign means gain or loss of function - the
# reading five pre-registered tests failed to support.
from piezo1.analysis.variant_impact import CouplingScore

print("attributes:", [f for f in CouplingScore.__dataclass_fields__])
print()
print("There is deliberately no `.direction` here, and no alias for it.")
print("A test enforces that, because the name was the misleading part.")